In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

import brain_probability_flux.plotting as bpp
from brain_probability_flux.dina.dynamics import ising as idyna

In [ ]:
# Results are read from data/. To plot a fresh run, set DATA_DIR to its run
# directory, for example Path("../data/2026-10-20").
DATA_DIR = bpp.DATA_DIR

In [ ]:
max_fig_width = bpp.FIG_WIDTH
max_fig_height = bpp.FIG_HEIGHT

In [ ]:
pfa, cluster_inds = bpp.aggregated_pfa(data_dir=DATA_DIR)
num_clusters = len(np.unique(cluster_inds))
color = bpp.spin_colors(num_clusters)
pfa.perform_WPCA(spin_colors=color)

In [ ]:
traffic_matrices = bpp.traffic_matrices_by_task(pfa)
undirected_indices = bpp.get_one_bit_difference_indices_undirected(
    num_clusters
)
inferred = bpp.load_results("inferred_networks", "arrhenius", DATA_DIR)

In [ ]:
# Traffic of the inferred Arrhenius model of each task, and its
# correlation with the observed traffic
model_traffic = [
    bpp.model_traffic_matrix(
        inferred["beta_J"][k],
        inferred["beta_h"][k],
        inferred["rate_const"][k],
        weight_type="arrhenius",
    )
    for k in range(bpp.NUM_TASKS)
]
r_values = [
    stats.pearsonr(
        traffic_matrices[k][undirected_indices],
        model_traffic[k][undirected_indices],
    ).statistic
    for k in range(bpp.NUM_TASKS)
]
fig, axs = plt.subplots(
    3, 3, figsize=(max_fig_width, max_fig_height), sharex=True, sharey=True
)
bpp.plot_pca_by_task(
    pfa,
    axs,
    kind="traffic",
    scale=20,
    color=color,
    arrows=[
        idyna.traffic_to_traffic_list(traffic) for traffic in model_traffic
    ],
    titles=[
        f"{name}, $r={r:.2f}$" for name, r in zip(pfa.task_name_list, r_values)
    ],
)
bpp.save_panel(fig, "../figures/fig-s23", "fig-s23", bbox_inches=None)
print(dict(zip(pfa.task_name_list, r_values)))